In [ ]:
from typing import Any
from __future__ import annotations

from IPython.display import display, Markdown

import sys
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import ast
import isodate
import pandas as pd
import numpy as np

ROOT: Path = Path.cwd().parent
sys.path.append(str(ROOT))

import src.core.util_examplification as examplification
BASE_DIR = Path("../data/raw")

PALETTE: dict[str, str] = {
    "bg": "#0D0F1A",
    "panel": "#141726",
    "accent1": "#FF4D6D",
    "accent2": "#4CC9F0",
    "accent3": "#F4A261",
    "accent4": "#2EC4B6",
    "text": "#E8EAF0",
    "muted": "#6C7293",
}
datasets: dict[str, list[Any]] = {
    "videos": [],
    "comments": [],
    "channels": [],
}
def printf(text_format):
    display(Markdown(text_format))
    
for csv_file in BASE_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(csv_file)

        df["source_file"] = csv_file.name
        df["source_path"] = str(csv_file)

        id_dir = next(
            (part for part in csv_file.parts if part.startswith("id_")),
            None
        )
        df["collection_id"] = id_dir

        stem_parts = csv_file.stem.split("_")
        country: str = stem_parts[-1] if len(stem_parts) > 2 else None
        df["country"] = country

        filename = csv_file.name.lower()

        if "video" in filename:
            datasets["videos"].append(df)

        elif "comment" in filename:
            datasets["comments"].append(df)

        elif "channel" in filename:
            datasets["channels"].append(df)

    except Exception as e:
        print(f"Erro ao ler {csv_file}: {e}")

videos_df = pd.concat(datasets["videos"], ignore_index=True) \
    if datasets["videos"] else pd.DataFrame()

comments_df = pd.concat(datasets["comments"], ignore_index=True) \
    if datasets["comments"] else pd.DataFrame()

channels_df = pd.concat(datasets["channels"], ignore_index=True) \
    if datasets["channels"] else pd.DataFrame()

print(f"Comments: {len(comments_df)}")
print(f"Channels: {len(channels_df)}")


In [ ]:

printf("# visão geral e estrutura do dataset")
printf("## amostra dos dados (primeiras 5 linhas)")
display(comments_df.head())

display(Markdown("---"))

printf("##  rsumo estatístico (variáveis nméricas)")
display(comments_df.describe()) 

if comments_df.select_dtypes(include=['object']).shape[1] > 0:
    printf("## resumo estatístico (variáveis categóricas)")
    display(comments_df.describe())